# Analyze many-seeds EM runs

Analysis of the 100-run EM clustering batches produced by `scripts/run_many_seeds.py`.

Per tumor, two files are loaded:
- `data/<tumor>/<tumor>_em_many_seeds_assignments.csv` — wide: one row per cell, columns `run_000`..`run_099`, values are cluster labels (`A11`, `A2212`, ...)
- `data/<tumor>/<tumor>_em_many_seeds_assignments.summary.csv` — per-run: seed, n_clusters, cluster_sizes

**Change `tumor` in the parameters cell and re-run the notebook to analyze a different tumor.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_context("notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

In [ ]:
# ---- Parameters: change `tumor` and re-run the whole notebook ----
tumor = "t57"

# 100-run batch (deterministic seeds)
# assign_path = Path(f"../data/{tumor}/{tumor}_em_many_seeds_assignments.csv")
# summary_path = Path(f"../data/{tumor}/{tumor}_em_many_seeds_assignments.summary.csv")
# coclust_path = Path(f"../data/{tumor}/{tumor}_em_many_seeds_coclustering.npy")  # 100-run

# 500-run batch (truly random seeds)
assign_path = Path(f"../data/{tumor}/{tumor}_em_many_seeds_500runs_assignments.csv")
summary_path = Path(f"../data/{tumor}/{tumor}_em_many_seeds_500runs_assignments.summary.csv")
coclust_path = Path(f"../data/{tumor}/{tumor}_em_many_seeds_500runs_coclustering.npy")  # 500-run

assignments = pd.read_csv(assign_path, index_col=0)  # index = cell, columns = runs
summary = pd.read_csv(summary_path)

run_cols = list(assignments.columns)
print(f"Tumor: {tumor}")
print(f"Assignments: {assignments.shape[0]} cells x {assignments.shape[1]} runs")
print(f"Summary: {summary.shape}")

In [ ]:
assignments.head()

## 1. Run-level overview

How many clusters does each run produce, and how variable is that across seeds?

In [ ]:
failed = summary[summary["n_clusters"] == -1]
print(f"Failed runs: {len(failed)}")
summary["n_clusters"].value_counts().sort_index()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
summary["n_clusters"].plot.hist(
    bins=range(1, int(summary["n_clusters"].max()) + 2),
    ax=ax, rwidth=0.9, align="left",
)
ax.set_xlabel("Number of clusters in a run")
ax.set_ylabel("Number of runs")
ax.set_title(f"{tumor}: cluster-count distribution across {len(summary)} runs")
plt.show()

## 2. Cell-level assignment stability

For each cell: how many distinct labels does it receive across runs, and how often does its most common (modal) label occur? A cell that always lands in the same cluster has modal frequency = 1.0; a cell that flips around has a lower value.

Note: labels like `A11` encode the recursion path, so two runs may use different label strings for biologically equivalent clusters — treat absolute stability values with care; relative comparisons between cells are more meaningful.

In [ ]:
n_runs = len(run_cols)

n_distinct = assignments.nunique(axis=1)
modal_freq = assignments.apply(lambda row: row.value_counts().iloc[0] / n_runs, axis=1)

stability = pd.DataFrame({"n_distinct_labels": n_distinct, "modal_freq": modal_freq})
print(f"Cells always assigned the same label: {(n_distinct == 1).sum()} / {len(stability)}")
stability.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(stability["n_distinct_labels"], bins=range(1, int(stability["n_distinct_labels"].max()) + 2), rwidth=0.9, align="left")
axes[0].set_xlabel("Distinct labels per cell")
axes[0].set_ylabel("Cells")
axes[1].hist(stability["modal_freq"], bins=20)
axes[1].set_xlabel("Modal-label frequency per cell")
axes[1].set_ylabel("Cells")
fig.suptitle(f"{tumor}: cell assignment stability across {n_runs} runs")
plt.tight_layout()
plt.show()

In [ ]:
# Most and least stable cells
print("Least stable cells:")
display(stability.sort_values("modal_freq").head(10))
print("Most stable cells:")
display(stability.sort_values("modal_freq", ascending=False).head(10))

## 3. Co-clustering matrix

For each pair of cells, the fraction of runs in which they share the same cluster label. This is label-agnostic (unlike raw label stability), so it is the most robust summary of the 100 runs: cells with co-clustering ≈ 1 are always together, ≈ 0 never together.

Computation note: this is an $n_{cells}^2$ operation. For ~2500 cells and 100 runs it takes a couple of minutes; the matrix is cached to disk so re-running the notebook is fast. Delete the cache file to recompute.

In [ ]:


if coclust_path.exists():
    coclust = np.load(coclust_path)
    print(f"Loaded cached co-clustering matrix {coclust.shape}")
else:
    # Factorize labels to integer codes per run, then accumulate co-membership
    codes = assignments.apply(lambda col: pd.factorize(col)[0]).to_numpy()  # cells x runs
    n_cells = codes.shape[0]
    coclust = np.zeros((n_cells, n_cells), dtype=np.float32)
    for j in range(codes.shape[1]):
        c = codes[:, j]
        coclust += (c[:, None] == c[None, :])
        if (j + 1) % 20 == 0:
            print(f"{j + 1}/{codes.shape[1]} runs")
    coclust /= codes.shape[1]
    np.save(coclust_path, coclust)
    print(f"Computed and cached co-clustering matrix {coclust.shape}")

cells = assignments.index.to_numpy()

In [ ]:
# Hierarchically clustered heatmap of the co-clustering matrix
# method/metric can be tuned; average linkage on 1 - coclust works well
g = sns.clustermap(
    coclust,
    method="average",
    metric="euclidean",
    cmap="viridis",
    xticklabels=False,
    yticklabels=False,
    figsize=(10, 10),
)
g.ax_heatmap.set_xlabel("cells")
g.ax_heatmap.set_ylabel("cells")
g.fig.suptitle(f"{tumor}: co-clustering frequency across {n_runs} runs", y=1.02)
plt.show()

# Keep the hierarchical ordering for later use
leaf_order = g.dendrogram_row.reordered_ind

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

# Hamming-like consensus distance
dist = np.rint((1 - coclust)*n_runs)
np.fill_diagonal(dist, 0)
condensed = squareform(dist, checks=False)

# Average linkage on the precomputed distance
Z = linkage(condensed, method="average")

# Clustered heatmap using this exact distance
g = sns.clustermap(
    dist,
    row_linkage=Z,
    col_linkage=Z,
    cmap="viridis_r",          # 0 = always together
    xticklabels=False,
    yticklabels=False,
    figsize=(10, 10),
)
g.ax_heatmap.set_xlabel("cells")
g.ax_heatmap.set_ylabel("cells")
g.fig.suptitle(f"{tumor}: Hamming distance across {n_runs} runs", y=1.02)
plt.show()

# Cut the dendrogram: e.g. group cells separated in < 20% of runs
cut_threshold = 10
hamming_clusters = pd.Series(
    fcluster(Z, t=cut_threshold, criterion="distance") - 1,
    index=cells,
    name="hamming_cluster",
)
print(f"Cut at d = {cut_threshold}: {hamming_clusters.nunique()} clusters")
print(hamming_clusters.value_counts().sort_index())

In [ ]:
# Consensus clusters: cut the co-clustering matrix at a threshold and extract groups
# (connected components of the graph where edges = co-clustering >= threshold)
consensus_threshold = 0.8

adjacency = coclust >= consensus_threshold
np.fill_diagonal(adjacency, False)

visited = np.zeros(len(cells), dtype=bool)
consensus_groups = []
for i in range(len(cells)):
    if visited[i]:
        continue
    # BFS from cell i
    stack = [i]
    group = []
    while stack:
        k = stack.pop()
        if visited[k]:
            continue
        visited[k] = True
        group.append(k)
        stack.extend(np.where(adjacency[k] & ~visited)[0])
    consensus_groups.append(sorted(group, key=lambda k: -coclust[k][group].sum()))

sizes = sorted((len(g) for g in consensus_groups), reverse=True)
print(f"Threshold {consensus_threshold}: {len(consensus_groups)} consensus groups")
print("Group sizes:", sizes)

# Map: cell -> consensus group id (ordered by size, group 0 = largest)
order = sorted(range(len(consensus_groups)), key=lambda g: -len(consensus_groups[g]))
cell_to_group = {}
for new_id, old_id in enumerate(order):
    for k in consensus_groups[old_id]:
        cell_to_group[cells[k]] = new_id
consensus_assignments = pd.Series(cell_to_group, name="consensus_group")
consensus_assignments.value_counts().sort_index()

In [ ]:
labels = fcluster(Z, t=0, criterion="distance")
from collections import Counter
#Counter sorted by frequency
Counter(labels)
Counter(labels).most_common()

In [ ]:
# Number of clusters with >=10 cells as a function of the cut threshold d
# d = max allowed Hamming distance within a cluster, in runs (0..n_runs)
# (d = 10 -> cells co-clustering in >= 90 of 100 runs are grouped)
thresholds = np.arange(0, n_runs + 1, 1)
min_size = 10

n_clusters = []
n_clusters_min = []
for t in thresholds:
    labels = fcluster(Z, t=t, criterion="distance")
    counts = np.bincount(labels)
    n_clusters.append(len(counts) - 1)  # bincount index 0 is unused (labels start at 1)
    n_clusters_min.append((counts[1:] >= min_size).sum())

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(thresholds, n_clusters, marker="o", markersize=3, label="all clusters")
# ax.plot(thresholds, n_clusters_min, marker="s", markersize=3,
#         label=f"clusters with ≥{min_size} cells")
ax.set_xlabel("Cut threshold d (runs in which cells may differ)")
ax.set_ylabel("Number of clusters")
ax.set_title(f"{tumor}: clusters vs. Hamming-distance cut")
#ax.set_ylim(0, 50)
ax.legend()
plt.show()

In [ ]:
# Number of clusters with >=10 cells as a function of the cut threshold d
# d = max allowed Hamming distance within a cluster, in runs (0..n_runs)
# (d = 10 -> cells co-clustering in >= 90 of 100 runs are grouped)
thresholds = np.arange(0, n_runs + 1, 1)
min_size = 10

n_clusters = []
n_clusters_min = []
for t in thresholds:
    labels = fcluster(Z, t=t, criterion="distance")
    counts = np.bincount(labels)
    n_clusters.append(len(counts) - 1)  # bincount index 0 is unused (labels start at 1)
    n_clusters_min.append((counts[1:] >= min_size).sum())

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(thresholds, n_clusters, marker="o", markersize=3, label="all clusters")
# ax.plot(thresholds, n_clusters_min, marker="s", markersize=3,
#         label=f"clusters with ≥{min_size} cells")
ax.set_xlabel("Cut threshold d (runs in which cells may differ)")
ax.set_ylabel("Number of clusters")
ax.set_title(f"{tumor}: clusters vs. Hamming-distance cut")
ax.set_ylim(0, 10)
ax.legend()
plt.show()

In [ ]:
# Number of clusters with >=10 cells as a function of the cut threshold d
# d = max allowed Hamming distance within a cluster, in runs (0..n_runs)
# (d = 10 -> cells co-clustering in >= 90 of 100 runs are grouped)
thresholds = np.arange(0, n_runs + 1, 1)
min_size = 10

n_clusters = []
n_clusters_min = []
for t in thresholds:
    labels = fcluster(Z, t=t, criterion="distance")
    counts = np.bincount(labels)
    n_clusters.append(len(counts) - 1)  # bincount index 0 is unused (labels start at 1)
    n_clusters_min.append((counts[1:] >= min_size).sum())

fig, ax = plt.subplots(figsize=(8, 4))
# ax.plot(thresholds, n_clusters, marker="o", markersize=3, label="all clusters")
ax.plot(thresholds, n_clusters_min, marker="s", markersize=3,
        label=f"clusters with ≥{min_size} cells")
ax.set_xlabel("Cut threshold d (runs in which cells may differ)")
ax.set_ylabel("Number of clusters")
ax.set_title(f"{tumor}: clusters vs. Hamming-distance cut")
#ax.set_ylim(0, 50)
ax.legend()
plt.show()

In [ ]:
# # Save consensus assignments alongside the other outputs
# out_path = Path(f"../data/{tumor}/{tumor}_em_many_seeds_consensus.csv")
# #consensus_assignments.to_csv(out_path, index_label="cell")
# print(f"Wrote {out_path}")